In [7]:
# 1. 라이브러리 설치
import sys
import os
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

# 경로 설정
sys.path.append(os.path.join(os.getcwd(), "Dashboard"))

from models import train_moa_model, train_toxicity_model
from data_loader import load_train_data, load_mappings

# 저장 폴더
BASE = Path(os.getcwd())
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)
    print(f"{MODEL_DIR} 폴더가 생성되었습니다.")

# 1. MoA 예측 모델 저장(TabPFN_drug_id_split 기준)
try:
    moa_pipe, moa_labels, g_cols = train_moa_model()
    joblib.dump(moa_pipe, f'{MODEL_DIR}/moa_model.pkl')
    joblib.dump({'labels': moa_labels, 'columns': g_cols}, f'{MODEL_DIR}/moa_metadata.pkl')
    print("MoA 모델 및 메타데이터 저장 完")
except Exception as e:
    print(f"MoA 모델 학습 중 오류 발생: {e}")


# 2. 세포독성 모델 학습 및 저장(Early_Toxicity_Screening 기준)
try:
    tox_pipe, _, c_cols = train_toxicity_model()
    joblib.dump(tox_pipe, f'{MODEL_DIR}/toxicity_model.pkl')
    joblib.dump(c_cols, f'{MODEL_DIR}/tox_metadata.pkl')
    print("세포독성 모델 및 컬럼 정보 저장 完")
except Exception as e:
    print(f"세포독성 모델 학습 중 오류 발생: {e}")

# 3. 폐암 특화 모델 저장(moa_outputs_lung)
try:
    # 폐암 세포주 id 리스트
    feat, _, _ = load_train_data()
    _, cell_info = load_mappings()
    lung_cells = cell_info[cell_info['ccle_name'].str.contains('LUNG', na=False)]['rid'].tolist()
    lung_c_cols = [c for c in lung_cells if c.startswith('c-')]


    # 폐암 특화 메타데이터 구성
    lung_metadata = {
        'target_tissue': 'Lung',
        'c_cols': lung_c_cols,          # 폐암 특화 세포주 컬럼 리스트
        'threshold': -2.0,              # 독성 판정 기준 수치
        'model_type': 'CatBoost/TabPFN' # 주요 사용 모델 타입
    }
    if 'lung_model' in locals():
        joblib.dump(lung_model, f'{MODEL_DIR}/lung_model.pkl')
        print("폐암 특화 모델 객체(.pkl) 저장 完")
    
    joblib.dump(lung_metadata, f'{MODEL_DIR}/lung_specific_metadata.pkl')
    print("폐암 특화 메타데이터 저장 完")

except Exception as e:
    print(f"폐암 특화 데이터 처리 중 오류 발생: {e}")

MoA 모델 및 메타데이터 저장 完
세포독성 모델 및 컬럼 정보 저장 完
폐암 특화 메타데이터 저장 完
